# BOS Flight Data Clean

Target：Estimate delay for at least 15 minutes（ArrDel15），and cancellation（Cancelled）。
Train set: 2022—2025
Test set: 2026 Jan — Jul


In [1]:
from pathlib import Path
import zipfile
import pandas as pd

#Data address
project_folder = Path.cwd()
save_folder = project_folder / "Data"
if (project_folder / "raw data").is_dir():
    save_folder = project_folder

raw_folder = save_folder / "raw data"
flight_folder = raw_folder / "bts"
weather_folder = raw_folder / "Weather"

# Find the folder that contains the annual CSVs, including a possible history subfolder.
matches = list(weather_folder.rglob("BOS_2022.csv"))
if len(matches) != 1:
    raise FileNotFoundError(f"Expected one BOS_2022.csv under {weather_folder}. Found: {matches}")
history_folder = matches[0].parent
print("Annual weather:", history_folder)
print("Clean CSVs:", save_folder)


low_visibility_limit = 1.0 # low visibility limit in miles
weather_time_limit = 120  # weather data time limit in minutes before departure

Annual weather: d:\大四\CS 506\Final Project\Data\raw data\Weather\history
Clean CSVs: d:\大四\CS 506\Final Project\Data


## 1. read 55 months flight data，keep data Origin == "BOS"


In [2]:
flight_columns = [
    "Year", "Month", "DayOfWeek", "FlightDate",
    "Reporting_Airline", "Flight_Number_Reporting_Airline",
    "Origin", "Dest", "CRSDepTime", "CRSArrTime",
    "Distance", "ArrDelay", "ArrDel15", "Cancelled", "Diverted"
]

monthly_tables = []
month_counts = []


for year in range(2022, 2027):
    if year == 2026:
        last_month = 7
    else:
        last_month = 12

    # Read the monthly flight data
    for month in range(1, last_month + 1):
        zip_file = flight_folder / f"bts_{year}_{month}.zip"
        csv_name = f"On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_{year}_{month}.csv"

        with zipfile.ZipFile(zip_file) as zip:
            with zip.open(csv_name) as csv_file:
                # read csv and record cols in flight_columns; regard as string for now
                month_flights = pd.read_csv(csv_file, usecols=flight_columns, dtype=str)

        # remove space and covert to upper case
        month_flights["Origin"] = month_flights["Origin"].str.strip().str.upper()

        # Keep if origin is BOS
        bos = month_flights[month_flights["Origin"] == "BOS"].copy()


        # Format string data into data frame date
        dates = pd.to_datetime(bos["FlightDate"], format="%Y-%m-%d")

        # Check if there is problem with date
        correct_date = (dates.dt.year == year) & (dates.dt.month == month)
        date_problems = len(bos) - correct_date.sum()

        # Merge data
        monthly_tables.append(bos)
        month_counts.append([year, month, len(bos), date_problems])

        print(year, month, "Number of Flights:", len(bos), "Date Problems:", date_problems, flush=True)

# Merge into one DataFrame
flights = pd.concat(monthly_tables, ignore_index=True)
# Create a DataFrame to store coverage information
coverage = pd.DataFrame(month_counts, columns=["Year", "Month", "BOSRows", "DateProblems"])

print("Number of Months:", len(coverage))
print("Total BOS Records:", len(flights))

del monthly_tables, month_flights, bos

2022 1 Number of Flights: 9917 Date Problems: 0
2022 2 Number of Flights: 9488 Date Problems: 0
2022 3 Number of Flights: 11073 Date Problems: 0
2022 4 Number of Flights: 11430 Date Problems: 0
2022 5 Number of Flights: 12116 Date Problems: 0
2022 6 Number of Flights: 11688 Date Problems: 0
2022 7 Number of Flights: 12179 Date Problems: 0
2022 8 Number of Flights: 12231 Date Problems: 0
2022 9 Number of Flights: 11569 Date Problems: 0
2022 10 Number of Flights: 12187 Date Problems: 0
2022 11 Number of Flights: 11638 Date Problems: 0
2022 12 Number of Flights: 10960 Date Problems: 0
2023 1 Number of Flights: 10871 Date Problems: 0
2023 2 Number of Flights: 10594 Date Problems: 0
2023 3 Number of Flights: 12339 Date Problems: 0
2023 4 Number of Flights: 12069 Date Problems: 0
2023 5 Number of Flights: 12428 Date Problems: 0
2023 6 Number of Flights: 11781 Date Problems: 0
2023 7 Number of Flights: 12133 Date Problems: 0
2023 8 Number of Flights: 12389 Date Problems: 0
2023 9 Number of Fl

## 2. Clean up text, and convert dates and numerical values

Use `strip` to remove leading and trailing whitespace, `upper` to standardize text to uppercase, convert to numeric format, and `replace` to treat empty strings as missing values.

In [3]:
# Clean and format the missing values
for column in flight_columns:
    flights[column] = flights[column].str.strip()
flights = flights.replace("", pd.NA)
flights = flights.replace("M", pd.NA)

for column in ["Reporting_Airline", "Origin", "Dest"]:
    flights[column] = flights[column].str.upper()

# Check for duplicate rows based on selected columns
flight_duplicate_count = flights.duplicated().sum()
print("Duplicate rows:", flight_duplicate_count)

# Convert data types for flightdate column
flights["FlightDate"] = pd.to_datetime(flights["FlightDate"], format="%Y-%m-%d")

for column in ["Distance", "ArrDelay", "ArrDel15", "Cancelled", "Diverted"]:

    # Numerical columns should be converted to numeric type, with errors coerced to NaN
    flights[column] = pd.to_numeric(flights[column], errors="coerce")

    # replace anomalous values 
    flights[column] = flights[column].replace([float("inf"), float("-inf")], float("nan"))

for column in ["ArrDel15", "Cancelled", "Diverted"]:
    # Check if the labels are only 0 or 1
    present_labels = flights[column].dropna()
    normal_count = present_labels.isin([0, 1]).sum()
    print(column, "Wrong labels: ", len(present_labels) - normal_count)

# Ensure that Distance is positive
flights.loc[flights["Distance"] <= 0, "Distance"] = float("nan")

# Check if ArrDel15 is consistent with ArrDelay
has_both = flights["ArrDelay"].notna() & flights["ArrDel15"].notna()
label_check = flights[has_both].copy()


expected_label = (label_check["ArrDelay"] >= 15).astype(int)
wrong_labels = (label_check["ArrDel15"] != expected_label).sum()

print("Inconsistent labels: ", wrong_labels)
del label_check
# count the number of missing values in each column
print(flights[flight_columns].isna().sum())

Duplicate rows: 0
ArrDel15 Wrong labels:  0
Cancelled Wrong labels:  0
Diverted Wrong labels:  0
Inconsistent labels:  0
Year                                   0
Month                                  0
DayOfWeek                              0
FlightDate                             0
Reporting_Airline                      0
Flight_Number_Reporting_Airline        0
Origin                                 0
Dest                                   0
CRSDepTime                             0
CRSArrTime                             0
Distance                               0
ArrDelay                           16325
ArrDel15                           16325
Cancelled                              0
Diverted                               0
dtype: int64


## 3. Convert scheduled time into a date-time value.

This process accounts for transitions related to daylight saving time (DST).


In [4]:
for column in ["CRSDepTime", "CRSArrTime"]:
    clock = pd.to_numeric(flights[column])
    hour = clock // 100  # 1441 // 100 = 14。
    minute = clock % 100  # 1441 % 100 = 41。

    normal_time = (hour >= 0) & (hour <= 23) & (minute >= 0) & (minute <= 59)
    valid_time = normal_time | (clock == 2400)

    print(column, "missing values", clock.isna().sum())
    print(column, "non-integer ", (clock % 1 != 0).sum())
    print(column, "invalid hour/minute values", len(clock) - valid_time.sum())

    if column == "CRSDepTime":
        departure_minutes = hour * 60 + minute
    else:
        flights["ArrHour"] = (hour % 24).astype(int)

# generate YYYY/MM/DD HH:MM datetime for departure
time_since_midnight = pd.to_timedelta(departure_minutes, unit="m")
flights["DepartureLocal"] = flights["FlightDate"] + time_since_midnight

flights["Year"] = flights["FlightDate"].dt.year
flights["Month"] = flights["DepartureLocal"].dt.month
flights["DayOfWeek"] = flights["DepartureLocal"].dt.day_of_week + 1  
flights["DepHour"] = flights["DepartureLocal"].dt.hour
flights["DepMinute"] = flights["DepartureLocal"].dt.minute

# Timezone conversion
flights["DepartureUTC"] = flights["DepartureLocal"].dt.tz_localize(
    "America/New_York", ambiguous="NaT", nonexistent="NaT"
)
flights["DepartureUTC"] = flights["DepartureUTC"].dt.tz_convert("UTC")
flights["DepartureUTC"] = flights["DepartureUTC"].astype("datetime64[ns, UTC]")

print("Anomoly Flights:", flights["DepartureUTC"].isna().sum())

CRSDepTime missing values 0
CRSDepTime non-integer  0
CRSDepTime invalid hour/minute values 0
CRSArrTime missing values 0
CRSArrTime non-integer  0
CRSArrTime invalid hour/minute values 0
Anomoly Flights: 0


## 4. Separate the tasks for "arrival delays" and "cancellations."

For the arrival delay task, apply the following filters: exclude cancelled flights, exclude diverted flights, and ensure the label is either 0 or 1.

For the cancellation task, only the validity of the cancellation label is required; do not remove flights simply because they lack an arrival label.

In [5]:
# Keep one row ID across CSVs produced by this run.
flights["FlightRowID"] = flights.index
delay_flights = flights[flights["Cancelled"] == 0].copy()
delay_flights = delay_flights[delay_flights["Diverted"] == 0].copy()
# Save both 0 and 1 for modeling before selecting positive cases for EDA.
delay_sample = delay_flights[delay_flights["ArrDel15"].isin([0, 1])].copy()
cancel_sample = flights[flights["Cancelled"].isin([0, 1])].copy()

delay_flights = delay_flights[delay_flights["ArrDel15"] == 1].copy()
cancel_flights = flights[flights["Cancelled"] == 1].copy()

print("Flights: ", len(flights))
print("Delay Flights: ", len(delay_flights))
print("Cancel Flights: ", len(cancel_flights))

Flights:  647451
Delay Flights:  142664
Cancel Flights:  14576


## 5. Read and clean weather data

tmpf = temperature (°F),   
relh = relative humidity (%),  
sknt = wind speed (knots),  
gust = wind gust (knots), 
vsby = visibility (miles),  
skyc1 = cloud category for the first layer.  
Missing gust values ​​are left as-is; they are not replaced with 0.

In [6]:
weather_columns = ["station", "valid", "tmpf", "relh", "sknt", "gust", "vsby", "skyc1"]
# Same observations and columns as before, now stored in five annual BOS files.
weather_tables = []
for year in range(2022, 2027):
    weather_file = history_folder / f"BOS_{year}.csv"
    year_weather = pd.read_csv(weather_file, usecols=weather_columns, dtype=str, na_values=["M"])
    weather_tables.append(year_weather)
weather = pd.concat(weather_tables, ignore_index=True)

weather_raw_count = len(weather)

# Clean and format the missing values
for column in weather_columns:
    weather[column] = weather[column].str.strip()
weather = weather.replace("", pd.NA)
weather["station"] = weather["station"].str.upper()
weather["skyc1"] = weather["skyc1"].str.upper()

bos_weather_count = (weather["station"] == "BOS").sum()
print("Check anomoly station", len(weather) - bos_weather_count)

# Clean and format weather date data
weather["valid"] = pd.to_datetime(weather["valid"], format="%Y-%m-%d %H:%M", utc=True)
print("Check anomoly weather time", weather["valid"].isna().sum())
weather["valid"] = weather["valid"].astype("datetime64[ns, UTC]")

# Clean and format the numeric columns
for column in ["tmpf", "relh", "sknt", "gust", "vsby"]:
    weather[column] = pd.to_numeric(weather[column], errors="coerce")
    weather[column] = weather[column].replace([float("inf"), float("-inf")], float("nan"))

# Clean anomalous values for weather
weather.loc[weather["relh"] < 0, "relh"] = float("nan")
weather.loc[weather["relh"] > 100, "relh"] = float("nan")
for column in ["sknt", "gust", "vsby"]:
    weather.loc[weather[column] < 0, column] = float("nan")

cloud_types = ["CLR", "FEW", "SCT", "BKN", "OVC", "VV"]
known_clouds = weather["skyc1"].dropna()
normal_cloud_count = known_clouds.isin(cloud_types).sum()
print("Check anomoly cloud types", len(known_clouds) - normal_cloud_count)

# Create binary columns for low visibility and cloud types
weather["LowVisibility"] = 0.0
weather.loc[weather["vsby"] < low_visibility_limit, "LowVisibility"] = 1.0
weather.loc[weather["vsby"].isna(), "LowVisibility"] = float("nan")

for cloud in cloud_types:
    column = "Cloud_" + cloud
    weather[column] = 0.0
    weather.loc[weather["skyc1"] == cloud, column] = 1.0
    weather.loc[weather["skyc1"].isna(), column] = float("nan")

Check anomoly station 0
Check anomoly weather time 0
Check anomoly cloud types 0


## 6. Handling Duplicate Weather Observations for the Same Timestamp

Duplicate records exist for the same timestamp. An examination reveals that for each set of duplicates, one record is relatively complete, while the others contain significant amounts of missing data.

To resolve this, first sort the records by "fewer missing values ​​(priority)" and then by "higher source row number (priority)." Finally, remove duplicates based on the station and timestamp.


In [7]:
weather_values = ["tmpf", "relh", "sknt", "gust", "vsby", "skyc1"]

weather["WeatherSourceRow"] = range(1, len(weather) + 1)
weather_duplicate_count = weather.duplicated(subset=["station", "valid"]).sum()

# sort_values in order to sort by missing count (ascending) and original row number (descending), keeping the latest record for ties.
weather["MissingCount"] = weather[weather_values].isna().sum(axis=1)
weather = weather.sort_values(["MissingCount", "WeatherSourceRow"], ascending=[True, False])
weather = weather.drop_duplicates(subset=["station", "valid"], keep="first")

weather = weather.drop(columns=["MissingCount"])
weather = weather.sort_values("valid")
weather = weather.reset_index(drop=True)

print("Raw records:", weather_raw_count)
print("Duplicate records:", weather_duplicate_count)
print("Final records:", len(weather))
print("Missing values:")
print(weather[weather_values].isna().sum())


# Gnerate local time and filter by date range
weather["LocalTime"] = weather["valid"].dt.tz_convert("America/New_York")
weather = weather[weather["LocalTime"].dt.tz_localize(None) >= "2022-01-01"].copy()
weather = weather[weather["LocalTime"].dt.tz_localize(None) < "2026-08-01"].copy()
weather["Year"] = weather["LocalTime"].dt.year
weather["Month"] = weather["LocalTime"].dt.month
weather["Hour"] = weather["LocalTime"].dt.hour



Raw records: 40057
Duplicate records: 14
Final records: 40043
Missing values:
tmpf         0
relh         2
sknt        12
gust     32112
vsby         4
skyc1        0
dtype: int64


## 7. Matching actual weather conditions with flights for correlation analysis

In [8]:
analysis_flights = flights[flights["Year"] <= 2025].copy()

weather_for_join = weather[
    [
        "valid", "WeatherSourceRow",
        "tmpf", "relh", "sknt", "gust", "vsby", "skyc1",
        "LowVisibility"
    ]
].copy()

# Rename
weather_for_join.columns = [
    "WeatherUTC", "WeatherSourceRow",
    "ActualTemp", "ActualHumidity", "ActualWind",
    "ActualGust", "ActualVisibility", "ActualCloud",
    "ActualLowVisibility"
]

# 3. Sort by time
analysis_flights = analysis_flights.sort_values("DepartureUTC")
weather_for_join = weather_for_join.sort_values("WeatherUTC")

# Find cloest historical weather data for each flight before flight using merge_asof
actual_weather = pd.merge_asof(
    analysis_flights,
    weather_for_join,
    left_on="DepartureUTC",
    right_on="WeatherUTC",
    direction="backward",
    tolerance=pd.Timedelta(minutes=weather_time_limit)
)

# 4. Calculate the time difference between the flight departure and the weather observation
time_difference = (
    actual_weather["DepartureUTC"] - actual_weather["WeatherUTC"]
)
actual_weather["WeatherAgeMinutes"] = time_difference.dt.total_seconds() / 60

print("Fail merge", actual_weather["WeatherUTC"].isna().sum())

Fail merge 72


## 8. Save Data

In [9]:
save_folder.mkdir(parents=True, exist_ok=True)


flights.to_csv(save_folder / "flights_clean.csv", index=False)

weather.to_csv(save_folder / "weather_clean.csv", index=False)

actual_weather.to_csv(
    save_folder / "actual_weather_2022_2025.csv",
    index=False
)

delay_flights.to_csv(
    save_folder / "delay_flights_clean.csv",
    index=False
)

cancel_flights.to_csv(
    save_folder / "cancel_flights_clean.csv",
    index=False
)

# Model samples include positive and negative labels.
delay_sample.to_csv(save_folder / "delay_model_sample.csv", index=False)
cancel_sample.to_csv(save_folder / "cancel_model_sample.csv", index=False)

# coverage.to_csv(save_folder / "monthly_coverage.csv", index=False)


# save the summary report
report_names = [
    "Months",
    "AllFlights",
    "FlightDuplicatesBeforeAddingID",
    "DelayValidFlights",
    "CancellationValidFlights",
    "CancelledFlights",
    "DivertedFlights",
    "MissingArrivalLabel",
    "WeatherRaw",
    "WeatherClean",
    "WeatherDuplicateTimesBeforeCleaning",
    "WeatherDuplicateTimesAfterCleaning",
    "ActualWeatherFlights",
    "ActualWeatherUnmatched"
]

report_numbers = [
    len(coverage),
    len(flights),
    flight_duplicate_count,
    len(delay_sample),
    len(cancel_sample),
    (flights["Cancelled"] == 1).sum(),
    (flights["Diverted"] == 1).sum(),
    flights["ArrDel15"].isna().sum(),
    weather_raw_count,
    len(weather),
    weather_duplicate_count,
    weather.duplicated(subset=["station", "valid"]).sum(),
    len(actual_weather),
    actual_weather["WeatherUTC"].isna().sum()
]
report = pd.DataFrame({
    "Item": report_names,
    "Count": report_numbers
})

report.to_csv(save_folder / "quality_summary.csv", index=False)


# Save the missing value counts for each column in flights and weather
flight_missing = flights[flight_columns].isna().sum()
weather_missing = weather[weather_columns].isna().sum()

flight_missing.to_csv(
    save_folder / "flight_missing.csv",
    index_label="Column",
    header=["MissingCount"]
)

weather_missing.to_csv(
    save_folder / "weather_missing.csv",
    index_label="Column",
    header=["MissingCount"]
)



print(report)
print("Data saved to:", save_folder)

                                   Item   Count
0                                Months      55
1                            AllFlights  647451
2        FlightDuplicatesBeforeAddingID       0
3                     DelayValidFlights  631126
4              CancellationValidFlights  647451
5                      CancelledFlights   14576
6                       DivertedFlights    1749
7                   MissingArrivalLabel   16325
8                            WeatherRaw   40057
9                          WeatherClean   40043
10  WeatherDuplicateTimesBeforeCleaning      14
11   WeatherDuplicateTimesAfterCleaning       0
12                 ActualWeatherFlights  563342
13               ActualWeatherUnmatched      72
Data saved to: d:\大四\CS 506\Final Project\Data


## 9. Historical weather at booking
For each station, calendar month and local hour, compare the 1999–2021 and 2012–2021 observation windows. A frequency is the number of observed events divided by the number of valid observations for that feature. These windows assume the booking occurred after 2021. Observations from the flight years are used for the earlier EDA only.

In [ ]:
# 9A. Turn the raw weather measurements into five event indicators.
risk_columns = ["Wind20", "VisBelow3", "CeilingBelow1000", "Snow", "Thunder"]
stations = ["BOS", "DCA", "LGA", "JFK", "ORD", "PHL"]
zones = {station: "America/New_York" for station in stations}
zones["ORD"] = "America/Chicago"

def read_snow_thunder(report):
    snow = 0
    thunder = 0
    found_time = False
    for word in str(report).upper().split():
        if word == "RMK":  # The rest is remarks.
            break
        if len(word) == 7 and word[:6].isdigit() and word.endswith("Z"):
            found_time = True  # Example: 191254Z
            continue
        if not found_time:  # Skip the station name and other header words.
            continue

        code = word.lstrip("+-")  # -SHSN becomes SHSN.
        if code in ["NIL", "//", "UP"]:
            return None, None
        if not code.isalpha():
            continue

        if code.startswith(("TS", "VCTS")):
            thunder = 1
        for i in range(0, len(code), 2):  # Read SHSN as SH, then SN.
            if code[i:i + 2] == "SN":
                snow = 1

    if not found_time:
        return None, None
    return snow, thunder


def weather_events(obs):
    # Keep missing measurements as missing so they do not count as good weather.
    wind = pd.to_numeric(obs["sknt"], errors="coerce")
    visibility = pd.to_numeric(obs["vsby"], errors="coerce")
    obs["Wind20"] = (wind >= 20).where(wind >= 0)
    obs["VisBelow3"] = (visibility < 3).where(visibility >= 0)

    # Check all four cloud layers for a broken/overcast ceiling below 1,000 feet.
    has_sky = pd.Series(False, index=obs.index)
    low_ceiling = has_sky.copy()
    unknown_height = has_sky.copy()
    for layer in range(1, 5):
        cover = obs[f"skyc{layer}"]
        height = pd.to_numeric(obs[f"skyl{layer}"], errors="coerce")
        has_sky |= cover.isin(["CLR", "SKC", "FEW", "SCT", "BKN", "OVC", "VV"])
        ceiling = cover.isin(["BKN", "OVC", "VV"])
        low_ceiling |= ceiling & (height >= 0) & (height < 1000)
        unknown_height |= ceiling & ~(height >= 0)
    obs["CeilingBelow1000"] = low_ceiling.where(low_ceiling | (has_sky & ~unknown_height))

     # Read each METAR and store one Snow and one Thunder value per row.
    flags = [read_snow_thunder(report) for report in obs["metar"]]
    obs[["Snow", "Thunder"]] = pd.DataFrame(flags, columns=["Snow", "Thunder"], index=obs.index)
    return obs[risk_columns].astype(float)

In [11]:
# 9B. Read one station and year at a time; keep counts instead of all raw reports.
annual_tables = []
for station in stations:
    for year in range(1999, 2022):
        weather_file = history_folder / f"{station}_{year}.csv"
        obs = pd.read_csv(weather_file, na_values=["M"])
        obs = obs[obs["station"] == station].copy()
        utc = pd.to_datetime(obs["valid"], utc=True, errors="coerce")
        local = utc.dt.tz_convert(zones[station])

        events = weather_events(obs)
        events["Month"] = local.dt.month
        events["Hour"] = local.dt.hour
        events["ReportTime"] = utc
        events["UTCHour"] = utc.dt.floor("h")
        events["MissingCount"] = events[risk_columns].isna().sum(axis=1)
        events = events[local.dt.year == year]

        # As in section 6, prefer the most complete report. Count each observed hour once.
        events = events.sort_values(["MissingCount", "ReportTime"], ascending=[True, False])
        events = events.drop_duplicates("UTCHour")
        groups = events.groupby(["Month", "Hour"])[risk_columns]
        counts = groups.count().add_suffix("_n").join(groups.sum().add_suffix("_events"))
        counts["Station"] = station
        counts["Year"] = year
        annual_tables.append(counts.reset_index())
    print("Finished historical weather:", station)

annual = pd.concat(annual_tables, ignore_index=True)

Finished historical weather: BOS
Finished historical weather: DCA
Finished historical weather: LGA
Finished historical weather: JFK
Finished historical weather: ORD
Finished historical weather: PHL


In [12]:
# 9C. Compute both historical windows from the same yearly counts.
climate_tables = []
for first_year in [1999, 2012]:
    selected = annual[annual["Year"] >= first_year].drop(columns="Year")
    totals = selected.groupby(["Station", "Month", "Hour"]).sum().reset_index()
    for column in risk_columns:
        valid = totals[column + "_n"]
        totals[column] = totals[column + "_events"] / valid.where(valid > 0)
    totals["FirstYear"] = first_year
    climate_tables.append(totals)

climate = pd.concat(climate_tables, ignore_index=True)
climate.to_csv(save_folder / "climate_month_hour.csv", index=False)
print(climate.groupby(["FirstYear", "Station"])["Wind20_n"].sum())

FirstYear  Station
1999       BOS        200774
           DCA        199225
           JFK        201182
           LGA        201104
           ORD        195517
           PHL        200527
2012       BOS         87493
           DCA         87372
           JFK         87588
           LGA         87502
           ORD         87483
           PHL         87581
Name: Wind20_n, dtype: int64


In [13]:
# 9D. Find the date of arrival using the scheduled local clock, including overnight flights.
def minutes_from_clock(values):
    clock = pd.to_numeric(values, errors="coerce")
    hour = clock // 100
    minute = clock % 100
    valid = (hour.between(0, 23) & minute.between(0, 59)) | (clock == 2400)
    return (hour * 60 + minute).where(valid & (clock % 1 == 0))

def local_arrival_utc(departure_utc, arrival_clock, zone):
    # These domestic routes take under 24 hours. Try the departure date in the destination zone.
    local_date = departure_utc.dt.tz_convert(zone).dt.tz_localize(None).dt.normalize()
    minutes = minutes_from_clock(arrival_clock) % 1440
    clock = local_date + pd.to_timedelta(minutes, unit="m")
    same_day = clock.dt.tz_localize(zone, ambiguous="NaT", nonexistent="NaT").dt.tz_convert("UTC")
    next_day_clock = clock + pd.Timedelta(days=1)
    next_day = next_day_clock.dt.tz_localize(zone, ambiguous="NaT", nonexistent="NaT").dt.tz_convert("UTC")
    return same_day.mask(same_day <= departure_utc, next_day)

booking = flights[["FlightRowID", "Origin", "Dest", "Month", "DepHour"]].copy()
booking["ArrivalMonth"] = pd.Series(pd.NA, index=booking.index, dtype="Int64")
booking["ArrivalHour"] = pd.Series(pd.NA, index=booking.index, dtype="Int64")
pilot_arrival = pd.Series(pd.NaT, index=flights.index, dtype="datetime64[ns, UTC]")
for dest in stations[1:]:
    use = flights["Dest"] == dest
    arrival = local_arrival_utc(flights.loc[use, "DepartureUTC"], flights.loc[use, "CRSArrTime"], zones[dest])
    pilot_arrival.loc[use] = arrival
    local = arrival.dt.tz_convert(zones[dest])
    booking.loc[use, "ArrivalMonth"] = local.dt.month.astype("Int64")
    booking.loc[use, "ArrivalHour"] = local.dt.hour.astype("Int64")

In [14]:
# 9E. Match origin weather to departure and destination weather to arrival.
weather_features = booking.copy()
for first_year in [1999, 2012]:
    lookup = climate[climate["FirstYear"] == first_year]
    for role in ["Origin", "Dest"]:
        if role == "Origin":
            month, hour = "Month", "DepHour"
        else:
            month, hour = "ArrivalMonth", "ArrivalHour"
        names = {"Station": role, "Month": month, "Hour": hour}
        for column in risk_columns:
            names[column] = f"{role}_{first_year}_{column}"
        lookup_columns = lookup[["Station", "Month", "Hour"] + risk_columns].rename(columns=names)
        weather_features = weather_features.merge(lookup_columns, on=[role, month, hour],
                                                  how="left", validate="many_to_one")

weather_features = weather_features.drop(columns=["Origin", "Dest", "Month", "DepHour", "ArrivalMonth", "ArrivalHour"])
assert weather_features["FlightRowID"].equals(flights["FlightRowID"])
weather_features.to_csv(save_folder / "booking_weather_features.csv", index=False)
print("Origin matches:", weather_features["Origin_1999_Wind20"].notna().sum(), "/", len(flights))
print("Destination matches:", weather_features["Dest_1999_Wind20"].notna().sum(), "/", flights["Dest"].isin(stations[1:]).sum())

Origin matches: 647451 / 647451
Destination matches: 163701 / 163701


## 10. BOS forecasts available six hours before departure
Use the latest available BOS airport forecast at the six-hour cutoff, then read the prevailing wind and visibility for the scheduled departure. Conditional TEMPO and PROB groups are excluded. Uncovered or cancelled forecasts have missing weather values.

In [15]:
# 10A. Convert a TAF day/hour into a UTC date (e.g. 0104 after January 31).
import re
from fractions import Fraction

def taf_time(token, reference):
    day = int(token[:2])
    hour = int(token[2:4])
    minute = int(token[4:] or 0)
    if hour > 24 or minute > 59 or (hour == 24 and minute != 0):
        return pd.NaT
    candidates = []
    for offset in [-1, 0, 1]:
        month = reference.normalize().replace(day=1) + pd.DateOffset(months=offset)
        if 1 <= day <= month.days_in_month:
            time = month + pd.Timedelta(days=day - 1, hours=hour, minutes=minute)
            candidates.append(time)
    if not candidates:
        return pd.NaT
    return min(candidates, key=lambda time: abs(time - reference))

In [16]:
# 10B. Decode one forecast period: sustained wind in knots and visibility below 3 miles.
def taf_values(text):
    # Only use the prevailing conditions. Stop before temporary or probability groups.
    text = re.split(r"\b(?:TEMPO|PROB\d{2}|BECMG|RMK)\b", text, maxsplit=1)[0]
    wind = re.search(r"\b(?:\d{3}|VRB)(\d{2,3})(?:G\d{2,3})?KT\b", text)
    vis = re.search(r"(?:^|\s)([PM]?)(\d+ \d+/\d+|\d+/\d+|\d+)SM\b", text)
    wind_speed = float("nan")
    low_visibility = float("nan")
    if wind:
        wind_speed = float(wind[1])
    if vis:
        # Examples: 1 1/2SM = 1.5 miles; P6SM = more than 6 miles.
        miles = sum(float(Fraction(piece)) for piece in vis[2].split())
        if vis[1] == "":
            low_visibility = float(miles < 3)
        elif vis[1] == "P" and miles >= 3:
            low_visibility = 0.0
        elif vis[1] == "M" and miles <= 3:
            low_visibility = 1.0
    return wind_speed, low_visibility

def taf_periods(text, archived):
    header = re.search(r"\bKBOS\s+(\d{6})Z\s+(?:(\d{4})/(\d{4})\b)?", text)
    if not header:
        return archived, []
    issued = taf_time(header[1], archived)
    available = max(archived, issued) if pd.notna(issued) else archived
    body = text[header.end():].split("=")[0]
    if not header[2] or re.search(r"\b(?:CNL|NIL)\b", body):
        return available, []
    start, end = taf_time(header[2], available), taf_time(header[3], available)
    if pd.isna(start) or pd.isna(end):
        return available, []
    # FM means “from”. Each new FM starts a new prevailing period.
    parts = re.split(r"\bFM(\d{6})\b", body)
    times = [start] + [taf_time(token, start) for token in parts[1::2]] + [end]
    periods = []
    for i, group in enumerate(parts[::2]):
        if pd.isna(times[i]) or pd.isna(times[i + 1]) or times[i] >= times[i + 1]:
            return available, []
        wind_speed, low_visibility = taf_values(group)
        periods.append((times[i], times[i + 1], wind_speed, low_visibility))
    return available, periods

In [17]:
# 10C. Read the monthly BOS TAF ZIPs inside Weather.
forecast_paths = list(weather_folder.rglob("TAFBOS_2022_01.zip"))
if len(forecast_paths) != 1:
    raise FileNotFoundError(f"Expected one TAFBOS_2022_01.zip under {weather_folder}. Found: {forecast_paths}")
taf_folder = forecast_paths[0].parent
bulletins = {}
for month in pd.period_range("2021-12", "2026-07", freq="M"):
    archive_path = taf_folder / f"TAFBOS_{month.year}_{month.month:02d}.zip"
    with zipfile.ZipFile(archive_path) as archive:
        for entry in archive.infolist():
            if entry.is_dir():
                continue
            name = Path(entry.filename).stem
            stamp = name.split("_")[-1]
            archived = pd.to_datetime(stamp, format="%Y%m%d%H%M", utc=True)
            text = " ".join(archive.read(entry).decode("ascii", errors="replace").upper().split())
            available, periods = taf_periods(text, archived)
            # An unordered, conflicting revision or cancellation should yield missing weather.
            if available in bulletins and bulletins[available] != periods:
                bulletins[available] = []
            else:
                bulletins[available] = periods

issues = pd.DataFrame({"TAFAvailableUTC": sorted(bulletins)})
rows = []
for available, periods in bulletins.items():
    for start, end, wind, low_vis in periods:
        rows.append([available, start, end, wind, low_vis])
periods = pd.DataFrame(rows, columns=["TAFAvailableUTC", "FromUTC", "UntilUTC", "TAFWindKt_6h", "TAFVisBelow3_6h"])
if periods.empty:
    raise ValueError("No readable BOS forecast periods found in the TAF ZIPs.")
for column in ["TAFAvailableUTC", "FromUTC", "UntilUTC"]:
    periods[column] = periods[column].astype("datetime64[ns, UTC]")
issues["TAFAvailableUTC"] = issues["TAFAvailableUTC"].astype("datetime64[ns, UTC]")
print("Forecast bulletins:", len(issues))

Forecast bulletins: 13241


In [18]:
# 10D. Match each flight to the forecast available six hours before scheduled departure.
requests = flights[["FlightRowID", "DepartureUTC"]].dropna().copy()
requests["CutoffUTC"] = requests["DepartureUTC"] - pd.Timedelta(hours=6)
latest = pd.merge_asof(requests.sort_values("CutoffUTC"), issues.sort_values("TAFAvailableUTC"),
                       left_on="CutoffUTC", right_on="TAFAvailableUTC", direction="backward")
latest = latest.dropna(subset=["TAFAvailableUTC"])
matched = pd.merge_asof(latest.sort_values("DepartureUTC"), periods.sort_values("FromUTC"),
                        by="TAFAvailableUTC", left_on="DepartureUTC", right_on="FromUTC", direction="backward")

# The selected period must still cover the departure. Preserve unmatched flights in the export.
valid = matched["DepartureUTC"] < matched["UntilUTC"]
forecast_columns = ["TAFWindKt_6h", "TAFVisBelow3_6h"]
matched.loc[~valid, forecast_columns] = float("nan")
assert (matched["TAFAvailableUTC"] <= matched["CutoffUTC"]).all()
forecast = flights[["FlightRowID"]].merge(matched[["FlightRowID", "TAFAvailableUTC"] + forecast_columns],
                                         on="FlightRowID", how="left", validate="one_to_one")
forecast = forecast.rename(columns={"TAFAvailableUTC": "TAFAvailableUTC_6h"})
forecast.to_csv(save_folder / "bos_taf_6h_features.csv", index=False)
print("BOS forecasts with wind:", forecast["TAFWindKt_6h"].notna().sum(), "/", len(flights))

BOS forecasts with wind: 646912 / 647451


## 11. Scheduled arrivals at the five destinations
Read only the schedule columns in the original BTS ZIPs. This second pass preserves the original BOS flight-reading cell. Count scheduled arrivals from all reporting origins in the inclusive ±30-minute window, including cancelled flights and the target flight itself. BTS provides a retrospective schedule proxy. The file does not record which schedule was published on the booking date.

In [19]:
# 11A. Convert national scheduled departures to UTC, then find the scheduled arrival.
airports = pd.read_csv(raw_folder / "airports.dat", header=None, usecols=[4, 11],
                       names=["Origin", "Zone"], na_values=[r"\N"])
airports = airports.dropna().drop_duplicates("Origin")
origin_zones = airports.set_index("Origin")["Zone"].to_dict()
origin_zones.update(zones)

def national_arrivals(table):
    date = pd.to_datetime(table["FlightDate"], errors="coerce")
    minutes = minutes_from_clock(table["CRSDepTime"])
    local_dep = date + pd.to_timedelta(minutes, unit="m")
    departure = pd.Series(pd.NaT, index=table.index, dtype="datetime64[ns, UTC]")
    for zone, rows in table.groupby(table["Origin"].map(origin_zones)).groups.items():
        local = local_dep.loc[rows].dt.tz_localize(zone, ambiguous="NaT", nonexistent="NaT")
        departure.loc[rows] = local.dt.tz_convert("UTC")

    # Use scheduled duration when available. Use the local clock if duration is missing/inconsistent.
    duration = pd.to_numeric(table["CRSElapsedTime"], errors="coerce")
    arrival = departure + pd.to_timedelta(duration.where(duration.between(1, 1440)), unit="m")
    for dest, rows in table.groupby("Dest").groups.items():
        local = arrival.loc[rows].dt.tz_convert(zones[dest])
        planned = minutes_from_clock(table.loc[rows, "CRSArrTime"]) % 1440
        wrong_clock = planned.notna() & (planned != local.dt.hour * 60 + local.dt.minute)
        repair = rows[arrival.loc[rows].isna() | wrong_clock]
        if len(repair) > 0:
            arrival.loc[repair] = local_arrival_utc(departure.loc[repair], table.loc[repair, "CRSArrTime"], zones[dest])
    return arrival

In [20]:
# 11B. Reopen each original ZIP; count schedules before filtering by origin.
schedule_columns = ["FlightDate", "Origin", "Dest", "CRSDepTime", "CRSArrTime", "CRSElapsedTime"]
arrival_tables = []
audit_rows = []
for year, month in coverage[["Year", "Month"]].itertuples(index=False, name=None):
    archive_path = flight_folder / f"bts_{year}_{month}.zip"
    csv_name = f"On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_{year}_{month}.csv"
    with zipfile.ZipFile(archive_path) as archive:
        with archive.open(csv_name) as csv_file:
            schedule = pd.read_csv(csv_file, usecols=schedule_columns, dtype=str)

    for column in ["Origin", "Dest"]:
        schedule[column] = schedule[column].str.strip().str.upper()
    schedule = schedule[schedule["Dest"].isin(stations[1:])].copy()
    schedule["ArrivalUTC"] = national_arrivals(schedule)
    counts = schedule.groupby(["Dest", "ArrivalUTC"]).size().rename("N")
    arrival_tables.append(counts.reset_index())
    for dest, group in schedule.groupby("Dest"):
        audit_rows.append([year, month, dest, len(group), group["ArrivalUTC"].notna().sum()])
    print("Finished scheduled arrivals:", year, month)

Finished scheduled arrivals: 2022 1
Finished scheduled arrivals: 2022 2
Finished scheduled arrivals: 2022 3
Finished scheduled arrivals: 2022 4
Finished scheduled arrivals: 2022 5
Finished scheduled arrivals: 2022 6
Finished scheduled arrivals: 2022 7
Finished scheduled arrivals: 2022 8
Finished scheduled arrivals: 2022 9
Finished scheduled arrivals: 2022 10
Finished scheduled arrivals: 2022 11
Finished scheduled arrivals: 2022 12
Finished scheduled arrivals: 2023 1
Finished scheduled arrivals: 2023 2
Finished scheduled arrivals: 2023 3
Finished scheduled arrivals: 2023 4
Finished scheduled arrivals: 2023 5
Finished scheduled arrivals: 2023 6
Finished scheduled arrivals: 2023 7
Finished scheduled arrivals: 2023 8
Finished scheduled arrivals: 2023 9
Finished scheduled arrivals: 2023 10
Finished scheduled arrivals: 2023 11
Finished scheduled arrivals: 2023 12
Finished scheduled arrivals: 2024 1
Finished scheduled arrivals: 2024 2
Finished scheduled arrivals: 2024 3
Finished scheduled arr

In [21]:
# 11C. Count arrivals in the 30 minutes before and after the target (including both endpoints).
counts = pd.concat(arrival_tables).groupby(["Dest", "ArrivalUTC"])["N"].sum()
traffic = flights[["FlightRowID", "Dest"]].copy()
traffic["ScheduledArrivalUTC"] = pilot_arrival
traffic["DestScheduledArrivals60min"] = pd.Series(pd.NA, index=traffic.index, dtype="Int64")
for dest in stations[1:]:
    if dest not in counts.index.get_level_values("Dest"):
        continue
    by_time = counts.loc[dest].sort_index()
    hourly_count = by_time.rolling("60min", center=True, closed="both").sum()
    # Leave the archive boundaries missing because the adjacent service dates are outside the data.
    use = (traffic["Dest"] == dest) & pilot_arrival.between("2022-01-03", "2026-07-31", inclusive="left")
    target_times = pilot_arrival.loc[use]
    traffic.loc[use, "DestScheduledArrivals60min"] = pd.array(hourly_count.reindex(target_times), dtype="Int64")

audit = pd.DataFrame(audit_rows, columns=["Year", "Month", "Dest", "ScheduledRows", "TimedRows"])
traffic.to_csv(save_folder / "destination_schedule_features.csv", index=False)
audit.to_csv(save_folder / "destination_schedule_coverage.csv", index=False)
print(audit.groupby("Dest")[["ScheduledRows", "TimedRows"]].sum())
print("Flights with a destination count:", traffic["DestScheduledArrivals60min"].notna().sum())

      ScheduledRows  TimedRows
Dest                          
DCA          647985     647985
JFK          556230     556229
LGA          709243     709243
ORD         1339789    1339458
PHL          435224     435223
Flights with a destination count: 163466
